## Downstream analysis of AlphaFold-Multimer output
#### This code is based on the code provided by the [VIB Tutorial](https://elearning.vib.be/courses/alphafold/)

In [ ]:
import glob
import math
import os
import numpy as np
from matplotlib import pyplot as plt
import argparse
import pickle
import json
import pandas as pd
from pandas import json_normalize

In [ ]:
input_dir = "/Volumes/Intenso/alphafold/models/vub_dimer_gpr37l1_model_20250812"

In [ ]:
feature_dict = pickle.load(open(f'{input_dir}/features.pkl','rb'))
model_names = sorted(glob.glob(f'{input_dir}/result_*.pkl'))

In [ ]:
def get_pae_plddt(model_names):
    out = {}
    for i,name in enumerate(model_names):
        d = pickle.load(open(name,'rb'))
        basename = os.path.basename(name)
        basename = basename[basename.index('model'):]
        out[f'{basename}'] = {
            "plddt": d["plddt"], ''
            "pae":d["predicted_aligned_error"], 
            "ptm":d["ptm"], 
            "iptm":d["iptm"]
        }
    return out

In [ ]:
pae_plddt_per_model = get_pae_plddt(model_names)

In [ ]:
models = sorted(pae_plddt_per_model.keys())
ptm   = np.array([pae_plddt_per_model[m]["ptm"]  for m in models], dtype=float)
iptm  = np.array([pae_plddt_per_model[m]["iptm"] for m in models], dtype=float)
rank  = 0.8 * iptm + 0.2 * ptm

data = np.vstack([ptm, iptm, rank])

metric_labels = ["pTM", "ipTM", "Ranking Score"]

plt.figure(figsize=(max(8, 0.8*len(models)), 3.8))
im = plt.imshow(data, aspect="auto", interpolation="nearest", vmin=0, vmax=1, cmap="magma")
plt.colorbar(im, label="Score (0–1)")

plt.yticks(range(len(metric_labels)), metric_labels)
plt.xticks(range(len(models)), models, rotation=45, ha="right")

plt.title("AlphaFold-Multimer: pTM, ipTM, and Ranking Score")
plt.tight_layout()

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        plt.text(j, i, f"{data[i, j]:.3f}", ha="center", va="center", color="white")

plt.show()

In [ ]:
msa = feature_dict['msa']
seqid = (np.array(msa[0] == msa).mean(-1))
seqid_sort = seqid.argsort()
non_gaps = (msa != 21).astype(float)
non_gaps[non_gaps == 0] = np.nan
final = non_gaps[seqid_sort] * seqid[seqid_sort, None]

In [ ]:
plt.figure(figsize=(14, 4), dpi=100)
plt.title("Sequence coverage")
plt.imshow(final, interpolation='nearest', aspect='auto', cmap="rainbow_r", vmin=0, vmax=1, origin='lower')
plt.plot((msa != 21).sum(0), color='black')
plt.xlim(-0.5, msa.shape[1] - 0.5)
plt.ylim(-0.5, msa.shape[0] - 0.5)
plt.colorbar(label="Sequence identity to query", )
plt.xlabel("Positions")
plt.ylabel("Sequences")

In [ ]:
plt.figure(figsize=(20, 15)) 
plt.title("Predicted LDDT per position")
for model_name, value in pae_plddt_per_model.items():
    plt.plot(value["plddt"], label=model_name)
plt.ylim(0, 100)
plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.ylabel("Predicted LDDT")
plt.xlabel("Positions")

In [ ]:
num_models =4 # columns
num_runs_per_model = math.ceil(len(model_names)/num_models)
fig = plt.figure(figsize=(3 * num_models, 2 * num_runs_per_model), dpi=100)
for n, (model_name, value) in enumerate(pae_plddt_per_model.items()):
    plt.subplot(num_runs_per_model, num_models, n + 1)
    plt.title(model_name)
    plt.imshow(value["pae"], label=model_name, cmap="bwr", vmin=0, vmax=30)
    plt.colorbar()
fig.tight_layout()